# 台股半導體量化交易系統 - 分析與測試

本 Notebook 用於互動式測試與分析量化交易策略

## 1. 環境設定與套件導入

In [ ]:
import sys
import os

# 加入父目錄到路徑
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 導入自定義模組
from config import BACKTEST_CONFIG, SEMICONDUCTOR_STOCKS
from src.data_loader import TaiwanStockDataLoader
from src.indicators import TechnicalIndicators
from src.strategies import StrategyManager
from src.backtester import Backtester
from src.visualizer import Visualizer

# 設定顯示選項
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# 設定中文字體
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 環境設定完成！")

## 2. 資料載入與預處理

In [ ]:
# 建立資料載入器
loader = TaiwanStockDataLoader()

# 設定回測期間（可自行調整）
start_date = '2022-01-01'
end_date = '2024-12-31'

# 下載並處理資料
data = loader.get_processed_data(start_date, end_date)

# 顯示資料摘要
summary = loader.get_data_summary()
print("\n資料摘要:")
display(summary)

## 3. 單一股票分析範例

In [ ]:
# 選擇要分析的股票（台積電）
symbol = '2330.TW'
stock_name = SEMICONDUCTOR_STOCKS[symbol]

print(f"分析股票: {stock_name} ({symbol})")

# 獲取資料
df = data[symbol].copy()

# 顯示基本統計
print(f"\n資料筆數: {len(df)}")
print(f"日期範圍: {df.index.min()} 至 {df.index.max()}")
print(f"\n價格統計:")
display(df[['open', 'high', 'low', 'close', 'volume']].describe())

## 4. 技術指標計算

In [ ]:
# 添加技術指標
df_indicators = TechnicalIndicators.add_all_indicators(df)

print("已添加的技術指標:")
indicator_cols = [col for col in df_indicators.columns if col not in df.columns]
print(indicator_cols)

# 顯示最新資料
print("\n最新 5 筆資料:")
display(df_indicators[['close', 'ma_short', 'ma_long', 'rsi', 'macd']].tail())

## 5. 策略測試與比較

In [ ]:
# 建立策略管理器
strategy_manager = StrategyManager()

# 比較所有策略
comparison = strategy_manager.compare_strategies(df_indicators)

print("策略訊號比較:")
display(comparison)

In [ ]:
# 測試混合策略
df_signals = strategy_manager.run_strategy('hybrid', df_indicators)

# 顯示訊號統計
buy_signals = (df_signals['signal'] == 1).sum()
sell_signals = (df_signals['signal'] == -1).sum()

print(f"買入訊號: {buy_signals} 個")
print(f"賣出訊號: {sell_signals} 個")

# 顯示訊號發生的日期
print("\n買入訊號日期:")
display(df_signals[df_signals['signal'] == 1][['close', 'rsi', 'macd']].head(10))

## 6. 回測執行

In [ ]:
# 建立回測引擎
backtester = Backtester(initial_capital=1_000_000)

# 執行回測
results = backtester.run_backtest(df_signals)

# 顯示績效指標
print("績效指標:")
display(backtester.get_metrics_summary())

In [ ]:
# 顯示交易記錄
if len(results['trades']) > 0:
    print("交易記錄:")
    display(results['trades'])
else:
    print("無交易記錄")

## 7. 視覺化分析

In [ ]:
# 建立視覺化器
visualizer = Visualizer()

# 繪製資金曲線
visualizer.plot_equity_curve(
    results['equity_curve'],
    title=f"{stock_name} 資金曲線"
)

In [ ]:
# 繪製交易訊號
visualizer.plot_signals(
    df_signals,
    title=f"{stock_name} 交易訊號"
)

In [ ]:
# 繪製技術指標
visualizer.plot_indicators(
    df_signals,
    title=f"{stock_name} 技術指標"
)

In [ ]:
# 繪製回撤分析
visualizer.plot_drawdown(
    results['equity_curve'],
    title=f"{stock_name} 回撤分析"
)

## 8. 參數優化實驗

In [ ]:
# 測試不同的 RSI 參數
from src.strategies import MeanReversionStrategy

rsi_params = [10, 14, 20, 30]
optimization_results = []

for rsi_period in rsi_params:
    # 重新計算 RSI
    df_test = df.copy()
    df_test['rsi'] = TechnicalIndicators.calculate_rsi(df_test['close'], rsi_period)
    df_test = TechnicalIndicators.add_all_indicators(df_test)
    
    # 生成訊號並回測
    df_test = strategy_manager.run_strategy('mean_reversion', df_test)
    
    backtester_test = Backtester(initial_capital=1_000_000)
    results_test = backtester_test.run_backtest(df_test)
    
    optimization_results.append({
        'RSI週期': rsi_period,
        '總報酬率': results_test['metrics']['總報酬率'],
        '夏普比率': results_test['metrics']['夏普比率'],
        '最大回撤': results_test['metrics']['最大回撤'],
        '交易次數': results_test['metrics']['交易次數']
    })

optimization_df = pd.DataFrame(optimization_results)
print("RSI 參數優化結果:")
display(optimization_df)

## 9. 多股票批次回測

In [ ]:
# 對所有股票執行回測
all_results = {}

for symbol, df_stock in data.items():
    print(f"\n回測 {SEMICONDUCTOR_STOCKS[symbol]} ({symbol})...")
    
    # 添加指標
    df_stock = TechnicalIndicators.add_all_indicators(df_stock)
    
    # 生成訊號
    df_stock = strategy_manager.run_strategy('hybrid', df_stock)
    
    # 回測
    backtester_stock = Backtester(initial_capital=1_000_000)
    results_stock = backtester_stock.run_backtest(df_stock)
    
    all_results[symbol] = results_stock
    
    print(f"  總報酬率: {results_stock['metrics']['總報酬率']:.2%}")
    print(f"  夏普比率: {results_stock['metrics']['夏普比率']:.2f}")

print("\n✅ 批次回測完成！")

In [ ]:
# 績效比較
comparison_data = []

for symbol, results in all_results.items():
    metrics = results['metrics']
    comparison_data.append({
        '股票代碼': symbol,
        '股票名稱': SEMICONDUCTOR_STOCKS[symbol],
        '總報酬率': metrics['總報酬率'],
        '年化報酬率': metrics['年化報酬率'],
        '夏普比率': metrics['夏普比率'],
        '最大回撤': metrics['最大回撤'],
        '勝率': metrics['勝率']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('總報酬率', ascending=False)

print("績效排名:")
display(comparison_df)

In [ ]:
# 繪製績效比較圖
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 總報酬率
axes[0, 0].barh(comparison_df['股票名稱'], comparison_df['總報酬率'])
axes[0, 0].set_title('總報酬率比較', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('報酬率')

# 夏普比率
axes[0, 1].barh(comparison_df['股票名稱'], comparison_df['夏普比率'])
axes[0, 1].set_title('夏普比率比較', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('夏普比率')

# 最大回撤
axes[1, 0].barh(comparison_df['股票名稱'], comparison_df['最大回撤'])
axes[1, 0].set_title('最大回撤比較', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('回撤')

# 勝率
axes[1, 1].barh(comparison_df['股票名稱'], comparison_df['勝率'])
axes[1, 1].set_title('勝率比較', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('勝率')

plt.tight_layout()
plt.show()

## 10. 結論與建議

根據回測結果,可以進一步:

1. 調整策略參數以優化績效
2. 測試不同的進出場條件
3. 加入更多風險管理機制
4. 考慮市場環境因素
5. 進行樣本外測試驗證

**注意**: 歷史績效不代表未來表現,實際交易請謹慎評估風險!